In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
import matplotlib.pyplot as plt
from arch import arch_model
import scipy
import statsmodels.api as sm
import importlib
import MarkovAutoregression_t

In [2]:
file_path = 'E:/RA/Geert/task1.py'
file_path = os.path.abspath(file_path)
dir_path = os.path.dirname(file_path)
os.chdir(dir_path)
excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter['Quarter_str'] = data_quarter['Year'].astype(str) + 'Q' + data_quarter['Quarter'].astype(str)
data_quarter.index = pd.PeriodIndex(data_quarter['Quarter_str'], freq='Q').to_timestamp()
data_month.index = pd.to_datetime(data_month['Year'].astype(str) + data_month['Month'].astype(str).str.zfill(2), format='%Y%m')
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock', 'Quarter_str']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year'] > 1969].copy()

In [3]:
sample_data['Inflation_lag_1'] = sample_data['Inflation'].shift(1)
sample_data['Inflation_lag_2'] = sample_data['Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] = sample_data['Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

## Regime Switching model with constraints

\begin{align}
    y_t = a_{S_t} + x_t' \beta_{S_t} + \phi_{1, S_t}
    (y_{t-1} - a_{S_{t-1}} - x_{t-1}' \beta_{S_{t-1}}) + \dots +
    \phi_{p, S_t} (y_{t-p} - a_{S_{t-p}} - x_{t-p}' \beta_{S_{t-p}}) +
    \varepsilon_t 
\end{align}

Use Kim Smoother and Hamilton filter to calculate marginal probability and joint probability

## Estimation Procedure

In regime-switching models, the likelihood surface is complex and riddled with numerous local maxima. Direct maximization of the likelihood function via standard Maximum Likelihood Estimation (MLE) methods can easily get trapped in one of these local maxima. So I apply following estimation procedure:

1. Random Search: performs a random search for better starting parameters. It draws random parameters around the initial start_params.

2. EM Algorithm: performs a specified number of EM algorithm iterations to further improve the initial parameters.

3. Maximum Likelihood Estimation (MLE): maximizes the likelihood function to find the best-fitting parameters. 

## Expectation-Maximization (EM) algorithm step

EM algorithm is designed to avoid getting stuck in local maxima by incrementally improving the likelihood at each step. EM smoothes the path towards more stable estimates that can be further refined using MLE methods.

### EM for Transition Probability Matrix

1. **Smoothed Joint Probabilities**: Calculate the smoothed probabilities for the transition from one regime to another using Kim Smoother and Hamilton Filter.

$$ P(S_{t-1} = i | Y) = \sum_{j=1}^{k} P(S_t = j, S_{t-1} = i | Y) $$

2. **Transition Probabilities Update**: Use the smoothed probabilities to update the transition probabilities.

$$ p_{ij} = \frac{\sum_{t=2}^{T} P(S_t = j, S_{t-1} = i | Y)}{\sum_{t=2}^{T} P(S_{t-1} = i | Y)} $$

Where $ p_{ij} $ is the probability of transitioning from regime $ i $ to regime $ j$, and $ T $ is the total number of observations.

3. **Normalization**: If the sum of probabilities exceeds 1 due to rounding errors, normalize the probabilities:

$$ p_{ij} \leftarrow \frac{p_{ij}}{\sum_{j=1}^{k} p_{ij}} $$

This ensures that the transition probabilities for each starting regime $ i $ sum to 1.

4. **Numerical Stability**: To avoid numerical instability, such as division by zero, a small constant $ \epsilon  $  is added:

$$ p_{ij} \leftarrow \frac{p_{ij}}{1 + \sum_{j=1}^{k} p_{ij} - 1 + \epsilon} $$


### EM for Exogenous Variable


### Non-Switching Coefficients

1. **Estimate Non-Switching Coefficients**:
   If there are coefficients that do not switch between regimes:

   $$ \text{coeffs}_{\text{non-switching}} = (X_{\text{non-switching}}'X_{\text{non-switching}})^{-1} X_{\text{non-switching}}'Y $$

   Here, $X_{\text{non-switching}} $is the design matrix of exogenous variables that do not switch, and $ Y $ is the vector of endogenous variables. These coefficients are assumed to be the same across all regimes.

### Switching Coefficients

2. **Preparation for Switching Coefficients**:
   For coefficients that do switch with regimes, the exogenous data is weighted by the square root of the smoothed marginal probabilities of being in each regime.

   $$ \tilde{Y} = \sqrt{P(S_t | Y)} \cdot Y $$
   $$ \tilde{X}_{\text{switching}} = \sqrt{P(S_t | Y)} \cdot X_{\text{switching}} $$

   where $ P(S_t | Y) $ are the smoothed marginal probabilities of each regime at time $ t $.

3. **Estimate Switching Coefficients**:
   Each regime's switching coefficients are estimated by applying the weighted least squares:

   $$ \text{coeffs}_{\text{switching}, i} = (\tilde{X}_{\text{switching}, i}' \tilde{X}_{\text{switching}, i})^{-1} \tilde{X}_{\text{switching}, i}' \tilde{Y}_i $$

   Here, the tilde denotes the weighted matrices and vectors, and $ i $ indexes the regime.


### EM for Autoregression

First, residuals are calculated for each regime. For each regime `i`:

1. Subtract the influence of exogenous variables from the original endogenous series, if applicable.
2. Create a lag matrix of the residuals, which serves as the regime-specific exogenous variables in the autoregressive process.
3. Use these lagged residuals to estimate the autoregressive coefficients by minimizing the weighted least squares problem.
4. If `switching_variance` is `True`, calculate the variance for each regime separately. Otherwise, compute a single variance across all regimes.

### Mathematical Details

- The residuals for each regime `i` are defined as:
  $$ \text{resid}_i = \text{orig_endog} - \text{orig_exog} \cdot \text{betas}[i] $$

- The autoregressive coefficients for each regime are estimated using the lagged values of the residuals:
  $$ \text{coeffs}[i] = (\text{tmp_exog}^T \cdot \text{tmp_exog})^{-1} \cdot \text{tmp_exog}^T \cdot \text{tmp_endog} $$

- The variance for each regime:
  $$ \text{variance}[i] = \frac{\sum (\text{tmp_resid}^2 \cdot \text{result.smoothed_marginal_probabilities}[i])}{\sum \text{result.smoothed_marginal_probabilities}[i]} $$

- If `switching_variance` is `False`, the combined variance across all regimes is calculated as the sum of squared residuals divided by the number of observations.


### EM for Variance

### Switching Variance

For each regime $i$:

1. Compute the residuals:
   - If exogenous variables are present, residuals are calculated as $ \text{resid}_i = endog - exog \cdot \text{betas}[i] $.
   - Otherwise, residuals are simply the endogenous variables.

2. Estimate the variance for each regime as:
   - $ \text{variance}[i] = \frac{\sum (\text{resid}_i^2 \cdot \text{smoothed_marginal_probabilities}[i])}{\sum \text{smoothed_marginal_probabilities}[i]} $

### Non-Switching Variance

If variances do not switch between regimes:

1. Optionally, weight the endogenous variables by the square root of the smoothed marginal probabilities.

2. Compute the weighted residuals for each regime:
   - If exogenous variables are present, adjust them with the weights and calculate the residuals.

3. The overall variance is then the weighted sum of squared residuals divided by the number of observations:
   - $ \text{variance} = \frac{\sum_{i} \text{resid}_i^2}{nobs} $


### EM for Nu

The process involves the following steps for each regime `i`:

1. Calculate residuals, which are the differences between the observed data and the model's predictions.
2. Compute the variance of these residuals.
3. Scale the residuals by their standard deviation.
4. Calculate the sample kurtosis of the scaled residuals.
5. Use the sample kurtosis to estimate `nu` by matching it with the theoretical kurtosis of the t-distribution.

If `switching_nu` is `True`, this process is done for each regime. Otherwise, a fixed `nu` is used, often set to 15 as a starting value in statistical software for robustness.

### Mathematical Details

- The residuals for each regime `i`:
  $$ \text{resid} = \text{endog} - \text{exog} \cdot \text{betas}[i] $$

- The variance for the residuals:
  $$ \text{variance} = \frac{\sum(\text{resid}^2 \cdot \text{smoothed_marginal_probabilities}[i])}{\sum \text{smoothed_marginal_probabilities}[i]} $$

- The sample kurtosis for scaled residuals is calculated, and the degrees of freedom `nu` is estimated by equating it with the theoretical kurtosis of the t-distribution:
  $$ \text{sample_kurt} = \text{kurtosis}(\text{scaled_resid}, \text{fisher=False}, \text{bias=False}) $$
  $$ \text{theoretical_kurt} = \frac{3 \cdot (\nu[i] - 2)}{(\nu[i] - 4)} $$
  $$ \nu[i] = \text{solve_for_nu}(\text{sample_kurt}, \text{theoretical_kurt}) $$


# Report 

For each mean model(AR,SPF), I report the following model results: 

| Error Distribution | #Regime | Switching AR | Switching SPF | Switching Distribution |
|--------------------|---------|--------------|---------------|------------------------|
| Normal             | 2       | Y            | Y             | Y                      |
| Student t          | 2       | Y            | Y             | Y                      |
| Normal             | 2       | Y            | N             | Y                      |
| Student t          | 2       | Y            | N             | Y                      |
| Normal             | 2       | N            | N             | Y                      |
| Student t          | 2       | N            | N             | Y                      |
| Normal             | 3      | Y            | N             | Y                      |
| Student t          | 3       | Y            | N             | Y                      |
| Normal             | 3       | N            | N             | Y                      |
| Student t          | 3       | N            | N             | Y                      |


In [57]:
def RS_results(endog, exog, order):
    switching_variance=True
    switching_nu=True
    trend="n"
    print('\n\n\nRS with Normal distribution, 2 regimes, switching AR, switching SPF and switching Distributional params\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =True,  switching_variance= switching_variance).fit()
    print(res1.summary())
    
    print('\n\n\n\nRS with Student t distribution, 2 regimes, switching AR, switching SPF and switching Distributional params\n')
    res2 = MarkovAutoregression_t.MarkovAutoregression_t( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =True,  switching_variance= switching_variance,switching_nu=switching_nu).fit()
    print(res2.summary())
    
    print('\n\n\n\nRS with Normal distribution, 2 regimes, switching AR  and switching Distributional params, without switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())
    
    print('\n\n\n\nRS with Student t distribution, 2 regimes, switching AR  and switching Distributional params, without switching SPF\n')
    res2 = MarkovAutoregression_t.MarkovAutoregression_t( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance,switching_nu=switching_nu).fit()
    print(res2.summary())
    
    print('\n\n\n\nRS with Normal distribution, 2 regimes,  switching Distributional params, without switching AR and switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, switching_ar=False,
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())
    
    print('\n\n\n\nRS with Student t distribution, 2 regimes, switching Distributional params, without switching AR  and switching SPF\n')
    res2 = MarkovAutoregression_t.MarkovAutoregression_t( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, switching_ar=False,
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance,switching_nu=switching_nu).fit()
    print(res2.summary())
    
    
    print('\n\n\n\nRS with Normal distribution, 3 regimes, switching AR  and switching Distributional params, without switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes=3,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())
    
    print('\n\n\n\nRS with Student t distribution, 3 regimes, switching AR  and switching Distributional params, without switching SPF\n')
    res2 = MarkovAutoregression_t.MarkovAutoregression_t( endog=endog,  exog=exog,  k_regimes= 3,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance,switching_nu=switching_nu).fit()
    print(res2.summary())
    
    print('\n\n\n\nRS with Normal distribution, 3 regimes,  switching Distributional params, without switching AR and switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 3,  order=order,  trend=trend, switching_ar=False,
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())
    
    print('\n\n\n\nRS with Student t distribution, 3 regimes, switching Distributional params, without switching AR  and switching SPF\n')
    res2 = MarkovAutoregression_t.MarkovAutoregression_t( endog=endog,  exog=exog,  k_regimes= 3,  order=order,  trend=trend, switching_ar=False,
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance,switching_nu=switching_nu).fit()
    print(res2.summary())

## Empirical Results

### Note:
In this version, I didn't apply EM for nu (student t paramters), and set starting values of nu = (4,5,...). Results shows that nu were stuck at starting values. And empty value(na) in the results are convergence Issues. MLE did not converge to a solution. Thus, in the future I will apply EM for nu (haven't figure out how to update nu using EM).

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [58]:
RS_results(endog= sample_data['Inflation shock'],exog=None, order=1)




RS with Normal distribution, 2 regimes, switching AR, switching SPF and switching Distributional params

                         Markov Switching Model Results                         
Dep. Variable:          Inflation shock   No. Observations:                  209
Model:             MarkovAutoregression   Log Likelihood                -173.496
Date:                  Wed, 08 Nov 2023   AIC                            358.991
Time:                          11:00:04   BIC                            379.045
Sample:                      07-01-1970   HQIC                           367.099
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2 

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:            Inflation shock   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -173.549
Date:                    Wed, 08 Nov 2023   AIC                            363.098
Time:                            11:00:05   BIC                            389.836
Sample:                        07-01-1970   HQIC                           373.908
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0801        nan        nan        nan         nan         nan
ar.L1         -0.021

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:            Inflation shock   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -174.074
Date:                    Wed, 08 Nov 2023   AIC                            364.148
Time:                            11:00:05   BIC                            390.886
Sample:                        07-01-1970   HQIC                           374.958
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0882        nan        nan        nan         nan         nan
ar.L1         -0.027

...

                          Markov Switching Model Results                          
Dep. Variable:            Inflation shock   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -175.906
Date:                    Wed, 08 Nov 2023   AIC                            365.813
Time:                            11:00:05   BIC                            389.209
Sample:                        07-01-1970   HQIC                           375.272
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0635        nan        nan        nan         nan         nan
nu             4.815

D:\anaconda\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                         Markov Switching Model Results                         
Dep. Variable:          Inflation shock   No. Observations:                  209
Model:             MarkovAutoregression   Log Likelihood                -168.214
Date:                  Wed, 08 Nov 2023   AIC                            360.427
Time:                          11:00:08   BIC                            400.535
Sample:                      07-01-1970   HQIC                           376.643
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0435      0.014      3.106      0.002       0.016       0.071
ar.L1          0.0079      0.225    

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:            Inflation shock   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -173.843
Date:                    Wed, 08 Nov 2023   AIC                            373.686
Time:                            11:00:14   BIC                            417.136
Sample:                        07-01-1970   HQIC                           391.253
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0498        nan        nan        nan         nan         nan
nu             5.004

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [59]:
RS_results(endog= sample_data['Inflation'],exog=sample_data['Forecasted inflation'], order=1)




RS with Normal distribution, 2 regimes, switching AR, switching SPF and switching Distributional params

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  209
Model:             MarkovAutoregression   Log Likelihood                -172.396
Date:                  Wed, 08 Nov 2023   AIC                            360.792
Time:                          11:00:18   BIC                            387.530
Sample:                      07-01-1970   HQIC                           371.602
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                                  Regime 0 parameters                                   
                           coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -170.995
Date:                    Wed, 08 Nov 2023   AIC                            361.989
Time:                            11:00:19   BIC                            395.413
Sample:                        07-01-1970   HQIC                           375.502
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                                  Regime 0 parameters                                   
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Forecasted inflation     0.9692      0.022     43.936      0.000     

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -171.609
Date:                    Wed, 08 Nov 2023   AIC                            361.218
Time:                            11:00:20   BIC                            391.299
Sample:                        07-01-1970   HQIC                           373.380
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.1419        nan        nan        nan         nan         nan
ar.L1         -0.025

...

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  209
Model:             MarkovAutoregression   Log Likelihood                -168.241
Date:                  Wed, 08 Nov 2023   AIC                            362.482
Time:                          11:00:23   BIC                            405.932
Sample:                      07-01-1970   HQIC                           380.049
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.1673      0.035      4.760      0.000       0.098       0.236
ar.L1          0.0596      0.080    

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -165.598
Date:                    Wed, 08 Nov 2023   AIC                            363.197
Time:                            11:00:26   BIC                            416.674
Sample:                        07-01-1970   HQIC                           384.818
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.1046        nan        nan        nan         nan         nan
ar.L1          0.102

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  209
Model:             MarkovAutoregression_t   Log Likelihood                -172.125
Date:                    Wed, 08 Nov 2023   AIC                            372.249
Time:                            11:00:29   BIC                            419.042
Sample:                        07-01-1970   HQIC                           391.168
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0693        nan        nan        nan         nan         nan
nu             4.920

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [61]:
RS_results(endog= sample_data['Inflation'],exog=sample_data['Forecasted inflation'], order=2)




RS with Normal distribution, 2 regimes, switching AR, switching SPF and switching Distributional params

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  208
Model:             MarkovAutoregression   Log Likelihood                -175.925
Date:                  Wed, 08 Nov 2023   AIC                            371.849
Time:                          11:00:54   BIC                            405.224
Sample:                      07-01-1970   HQIC                           385.344
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                                  Regime 0 parameters                                   
                           coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -169.584
Date:                    Wed, 08 Nov 2023   AIC                            363.169
Time:                            11:00:55   BIC                            403.219
Sample:                        07-01-1970   HQIC                           379.363
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                                  Regime 0 parameters                                   
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Forecasted inflation     0.9747      0.026     36.822      0.000     

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -168.378
Date:                    Wed, 08 Nov 2023   AIC                            358.757
Time:                            11:00:56   BIC                            395.470
Sample:                        07-01-1970   HQIC                           373.602
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.3832        nan        nan        nan         nan         nan
ar.L1         -0.007

...

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  208
Model:             MarkovAutoregression   Log Likelihood                -157.514
Date:                  Wed, 08 Nov 2023   AIC                            347.027
Time:                          11:01:03   BIC                            400.428
Sample:                      07-01-1970   HQIC                           368.620
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         1.8934      0.768      2.464      0.014       0.387       3.399
ar.L1          0.6742      0.214    

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -167.823
Date:                    Wed, 08 Nov 2023   AIC                            373.645
Time:                            11:01:10   BIC                            437.059
Sample:                        07-01-1970   HQIC                           399.286
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.3842        nan        nan        nan         nan         nan
ar.L1          0.064

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -184.222
Date:                    Wed, 08 Nov 2023   AIC                            398.444
Time:                            11:01:16   BIC                            448.507
Sample:                        07-01-1970   HQIC                           418.687
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.3679        nan        nan        nan         nan         nan
nu             5.662

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [60]:
RS_results(endog= sample_data['Inflation'],exog=sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']], order=2)




RS with Normal distribution, 2 regimes, switching AR, switching SPF and switching Distributional params

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  208
Model:             MarkovAutoregression   Log Likelihood                -172.474
Date:                  Wed, 08 Nov 2023   AIC                            368.949
Time:                          11:00:30   BIC                            408.999
Sample:                      07-01-1970   HQIC                           385.143
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                                     Regime 0 parameters                                      
                                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -165.397
Date:                    Wed, 08 Nov 2023   AIC                            358.795
Time:                            11:00:32   BIC                            405.520
Sample:                        07-01-1970   HQIC                           377.688
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                                     Regime 0 parameters                                      
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Forecasted inflation           0.7605      0.417   

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -169.049
Date:                    Wed, 08 Nov 2023   AIC                            362.099
Time:                            11:00:34   BIC                            402.149
Sample:                        07-01-1970   HQIC                           378.293
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.3943        nan        nan        nan         nan         nan
ar.L1          0.002

...

                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -182.039
Date:                    Wed, 08 Nov 2023   AIC                            384.079
Time:                            11:00:35   BIC                            417.454
Sample:                        07-01-1970   HQIC                           397.574
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.3717        nan        nan        nan         nan         nan
nu             5.076

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -167.458
Date:                    Wed, 08 Nov 2023   AIC                            374.916
Time:                            11:00:47   BIC                            441.667
Sample:                        07-01-1970   HQIC                           401.907
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.3888        nan        nan        nan         nan         nan
ar.L1          0.055

C:\Users\Administrator\MarkovAutoregression_t.py:103: ComplexWarning: Casting complex values to real discards the imaginary part
  #nu = nu.astype(np.float64)
C:\Users\Administrator\MarkovAutoregression_t.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  #variance = variance.astype(np.float64)


                          Markov Switching Model Results                          
Dep. Variable:                  Inflation   No. Observations:                  208
Model:             MarkovAutoregression_t   Log Likelihood                -181.553
Date:                    Wed, 08 Nov 2023   AIC                            395.107
Time:                            11:00:53   BIC                            448.508
Sample:                        07-01-1970   HQIC                           416.699
                             - 10-01-2022                                         
Covariance Type:                   approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.3699        nan        nan        nan         nan         nan
nu             4.553